In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI

openai_client = OpenAI()
openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

In [2]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

print(f"Loaded {sqlite_index.count()} documents from faq.db")

Loaded 153 documents from faq.db


In [3]:
from rag_helper import RAGBase

custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
    model=os.getenv("OPENAI_MODEL"),
    instructions=custom_instructions
)

In [4]:
print(assistant.rag("What can I learn?")) # Example question not working well whithout an agentic RAG approach.

Based on the provided context, the specific topics you can learn about are not explicitly listed. However, the context indicates that you can:

*   Use **Evidently** for the capstone project.
*   Use **Bluesky** for "learning in public" credits.
*   Work with **non-English datasets** for the capstone project (provided documentation is in English).
*   **Join the course** at any time, though you must submit your project while submissions are still open to receive a certificate.
*   **Evaluate projects** assigned to you after the submission deadline has passed.


In [5]:
# Cost calculation example

print(f"using model: {os.getenv('OPENAI_MODEL')}")

res = openai_client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL"),
    messages=[{"role": "user", "content": "Hi!"}]
)

from cost_calculator_helper import calculate_ik_gemma_price, print_cost_breakdown
input_tokens = res.usage.prompt_tokens
output_tokens = res.usage.completion_tokens

calculated_cost = calculate_ik_gemma_price(input_tokens, output_tokens)

print_cost_breakdown(calculated_cost)

using model: qwen3
Input Cost: CHF 0.000003
Output Cost: CHF 0.000004
Total Cost: CHF 0.000007


In [8]:

# Defining the tool for the agentic RAG approach

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return sqlite_index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

# define the tool schema corresponding to the search function
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query to find relevant FAQ entries."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}

messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

res = openai_client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL"),
    messages=messages,
    tools=[search_tool]
)

print(res.choices[0].message.content)

None


In [9]:

import json

function_args = json.loads(res.choices[0].message.tool_calls[0].function.arguments)

search_results = search(**function_args)

messages.append({
    "role": "tool",
    "call_id": res.choices[0].message.tool_calls[0].id,
    "content": json.dumps(search_results)
})

print(f"Search results: {search_results}")

res = openai_client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL"),
    messages=messages,
    tools=[search_tool]
)

print(res.choices[0].message.content)
print(res.usage.total_tokens)



Search results: [{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '5cc511f85b', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Does the course certificate show the number of course hours?', 'answer': 'No. The certificate does not state a total number of hours.'}, {'id': 'bd31146b0e', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'When will the course be offered next?', 'answer': 'Summer 2027.'}, {'id': '69d122f12e', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?', 'answer': 'No, you can only get a certificate if you finish the course with a "live"